<a href="https://colab.research.google.com/github/Notyourpatrick/Air-Quality-checker/blob/main/Real_Time_Global_Air_Quality_Dashboard_using_Dash_(by_Plotly).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install dash plotly pandas requests dash-bootstrap-components

In [ ]:
import requests
import pandas as pd
import numpy as np
import dash
from dash import html, dcc, dash_table  # Added dash_table here
from dash.dependencies import Input, Output
import plotly.express as px
import dash_bootstrap_components as dbc
from google.colab import output

# --- 1. DATA INGESTION ---
def fetch_air_quality_data():
    url = "https://api.openaq.org/v3/parameters/2/latest?limit=100"
    headers = {"accept": "application/json"}
    try:
        response = requests.get(url, headers=headers, timeout=5)
        if response.status_code == 200:
            data = response.json()
            records = []
            for result in data.get('results', []):
                coordinates = result.get('coordinates', {})
                if coordinates.get('latitude') and coordinates.get('longitude'):
                    records.append({
                        'City': result.get('name', 'Reporting Station'),
                        'Country': result.get('country', 'Global'),
                        'Latitude': float(coordinates.get('latitude')),
                        'Longitude': float(coordinates.get('longitude')),
                        'PM25_Value': float(result.get('value', 0))
                    })
            if records: return pd.DataFrame(records)
    except Exception:
        pass

    # Fallback simulation data
    np.random.seed(42)
    mock_cities = ['Mumbai', 'Delhi', 'New York', 'London', 'Tokyo', 'Sydney', 'Cairo', 'Paris', 'Beijing', 'Los Angeles']
    mock_countries = ['IN', 'IN', 'US', 'UK', 'JP', 'AU', 'EG', 'FR', 'CN', 'US']
    mock_lats = [19.07, 28.61, 40.71, 51.50, 35.67, -33.86, 30.04, 48.85, 39.90, 34.05]
    mock_lons = [72.87, 77.20, -74.00, -0.12, 139.65, 151.20, 31.23, 2.35, 116.40, -118.24]
    return pd.DataFrame([{
        'City': mock_cities[i], 'Country': mock_countries[i],
        'Latitude': mock_lats[i], 'Longitude': mock_lons[i],
        'PM25_Value': round(np.random.uniform(5, 145), 2)
    } for i in range(10)])

# --- 2. DASH APP SETUP ---
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.CYBORG])

app.layout = dbc.Container([
    dbc.Row([
        dbc.Col(html.H1("Live Global Air Quality Tracker", className="text-center my-4 text-info"), width=12)
    ]),
    dbc.Row([
        dbc.Col([
            html.Label("Select Minimum PM2.5 Threshold:", className="text-white"),
            dcc.Slider(id='pm-slider', min=0, max=150, step=10, value=0, marks={i: str(i) for i in range(0, 151, 25)}),
            html.Div(id='data-summary-card', className="text-white mt-3 p-3 border border-secondary rounded")
        ], md=4),

        dbc.Col([
            dcc.Graph(id='global-map', style={'height': '40vh'}),

            # NEW COMPONENT: This adds the empty spreadsheet box below our map
            html.Div([
                html.H5("Filtered Location Data Inventory", className="text-info mt-4 mb-2"),
                dash_table.DataTable(
                    id='data-grid-table',
                    columns=[
                        {"name": "City Location", "id": "City"},
                        {"name": "Country Code", "id": "Country"},
                        {"name": "PM2.5 Conc. (µg/m³)", "id": "PM25_Value"}
                    ],
                    page_size=5,  # limits it to 5 rows per page with built-in navigation buttons
                    style_header={'backgroundColor': '#1a1a1a', 'color': '#0dcaf0', 'fontWeight': 'bold'},
                    style_cell={'backgroundColor': '#333333', 'color': 'white', 'textAlign': 'left', 'padding': '8px'},
                )
            ])
        ], md=8)
    ])
], fluid=True)

# --- 3. INTERACTIVE CALLBACKS ---
@app.callback(
    [Output('global-map', 'figure'),
     Output('data-summary-card', 'children'),
     Output('data-grid-table', 'data')],
    [Input('pm-slider', 'value')]
)
def update_dashboard(min_pm_value):
    df = fetch_air_quality_data()
    filtered_df = df[df['PM25_Value'] >= min_pm_value]

    # 1. Handle Map Rendering
    if not filtered_df.empty:
        map_fig = px.scatter_mapbox(
            filtered_df, lat="Latitude", lon="Longitude", hover_name="City",
            color="PM25_Value", size="PM25_Value", size_max=15, zoom=1,
            mapbox_style="carto-darkmatter", color_continuous_scale=px.colors.sequential.Plasma
        )
    else:
        map_fig = px.scatter_mapbox(lat=[0], lon=[0], zoom=1, mapbox_style="carto-darkmatter")
    map_fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0}, paper_bgcolor="rgba(0,0,0,0)")

    # 2. Handle Text Summaries
    summary_text = [
        html.H5("Quick Insights", className="text-info"),
        html.P(f"Total Available Cities: {len(df)}"),
        html.P(f"Cities Displayed (Above Threshold): {len(filtered_df)}")
    ]

    # 3. NEW RETURN: Send data dictionary straight to the spreadsheet element rows
    table_records = filtered_df.to_dict('records')

    return map_fig, summary_text, table_records

# --- 4. COLAB SECURE EXECUTION ---
output.serve_kernel_port_as_iframe(8050, height="800")
app.run(host='0.0.0.0', port=8050)

<IPython.core.display.Javascript object>

Dash is running on http://0.0.0.0:8050/



INFO:dash.dash:Dash is running on http://0.0.0.0:8050/



 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8050
 * Running on http://172.28.0.12:8050
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [26/Jun/2026 14:17:49] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [26/Jun/2026 14:17:50] "GET /_dash-component-suites/dash/deps/prop-types@15.v4_3_0m1782482054.8.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [26/Jun/2026 14:17:50] "GET /_dash-component-suites/dash/html/dash_html_components.v4_3_0m1782482054.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [26/Jun/2026 14:17:50] "GET /_dash-component-suites/dash/dcc/dash_core_components-shared.v4_3_0m1782482054.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [26/Jun/2026 14:17:50] "GET /_dash-component-suites/dash/deps/react-dom@18.v4_3_0m1782482054.3.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [26/Jun/2026 14